# Subsample analysis — Alexi UC (pure, adata-only)

Reads the **self-contained analysis object** written by
`metab_processing/SpaceTravLR/run_subsamples.py`,
`spacetravlr_subsamples/run_{R}/subsample_betas.h5ad`, and does everything from it — **no
SpaceTravLR / torch, no re-diffusion**. That one AnnData carries:

- `X` — raw counts, and `obs[TIER]` — the cell-type annotation (it *is* the display adata),
- `obsm['beta_{gene}__sample{j}']` — the trained betas, read from the betadata **database**
  (parquets) at run time, one matrix per (subsample, target gene); `uns['subsample_index']`
  names each one's `metab@…` columns,
- `obsm['x_metab']` — the metabolite communication scores **x** (cells × gene pair),
  precomputed and stored by the run, with column names in `uns['x_metab_modulators']`.

Each surviving gene pair is one `metab@{Metabolite}-{g1}_{g2}` column (both orientations
summed). For a target gene and cell type (start with **`T`**) we compute: **(1)** average β per
gene pair, **(2)** the β distribution per gene pair, **(3)** the **R²** between the cell type's
raw counts of the gene and **β·x**. β·x = `beta[:, pair] * x_metab[:, pair]`.

> If `x_metab` is missing from the file, re-run so the run stores it (the job now computes and
> saves it); the notebook says so in the Diagnostic and the β·x / R² cells are skipped.

In [1]:
import sys
from pathlib import Path
# repo root on sys.path only so we can import the (pure) DATA_DIR constant. Nothing here
# imports SpaceTravLR/torch.
_start = Path.cwd()
_root = next((p for p in (_start, *_start.parents)
             if (p / ".git").exists() or (p / "setup.py").exists()), _start)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import json
import re
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from scipy.stats import linregress

from metab_processing.metab_travlr_config import DATA_DIR

## Settings

In [2]:
# ============================ SETTINGS ============================
PROJECT   = 'Alexi_UC_Spliced'
DATASET   = '13473_HS4_UC-Slice_1'   # which slice's subsample run to read
RUN       = -1                       # subsampling run number; -1 = latest
TIER      = '25_06_11_ICI_5K_Coarse_annotations'   # cell-type annotation column in obs
CELL_TYPE = 'T'                      # start here
GENE      = 'NDRG1'                  # primary target gene
RAW_LAYER = None                     # None -> adata.X is raw counts; else a layer name

dataset_dir  = Path(DATA_DIR) / PROJECT / DATASET
sub_root_all = dataset_dir / 'spacetravlr_subsamples'
_runs = sorted(int(m.group(1)) for p in sub_root_all.glob('run_*')
               if (m := re.match(r'run_(\d+)$', p.name)))
R = _runs[-1] if RUN == -1 else RUN
sub_root = sub_root_all / f'run_{R}'
h5 = sub_root / 'subsample_betas.h5ad'
print('dataset :', DATASET)
print('run     :', R, '->', sub_root)
assert h5.is_file(), f'no analysis adata at {h5} (available runs: {_runs})'

dataset : 13473_HS4_UC-Slice_1
run     : 1 -> /global/scratch/fsa/fc_wagnerlabfca/fosterangus/MetabTravLR/Data/Alexi_UC_Spliced/13473_HS4_UC-Slice_1/spacetravlr_subsamples/run_1


## Load — one adata, everything in it

In [3]:
adata = sc.read_h5ad(h5)
adata.uns['sample'] = DATASET
SUBIDX = json.loads(adata.uns['subsample_index'])          # [{key, sample, gene, columns}, ...]
SAMPLES = sorted({e['sample'] for e in SUBIDX})
GENES   = sorted({e['gene'] for e in SUBIDX})
HAS_X   = 'x_metab' in adata.obsm
print('adata     :', adata.shape)
print('subsamples:', SAMPLES)
print('target genes with betas:', GENES)

# x_metab (cells x gene-pair) as a labelled DataFrame; empty frame if the run did not store it.
if HAS_X:
    X = pd.DataFrame(adata.obsm['x_metab'], index=adata.obs_names,
                     columns=list(adata.uns['x_metab_modulators']))
    print('x_metab   :', X.shape, '(gene-pair columns)')
else:
    X = pd.DataFrame(index=adata.obs_names)
    print('x_metab   : MISSING -- re-run so the job stores it; beta*x / R^2 cells will skip.')

assert TIER in adata.obs, f'{TIER!r} not in adata.obs; e.g. {list(adata.obs.columns)[:8]}'
print('cell types:', sorted(map(str, pd.unique(adata.obs[TIER].astype(str)))))

adata     : (12142, 31)
subsamples: [1, 2, 3, 4, 5, 6]
target genes with betas: ['ATP1A1', 'CD3E', 'CD3G', 'CD4', 'CSF2', 'CTLA4', 'ENTPD1', 'FOXP3', 'HAVCR2', 'HIF1A', 'IL10', 'IL17A', 'IL17F', 'IL23R', 'IL2RA', 'LAG3', 'MAPK11', 'MAPK12', 'MAPK14', 'MYC', 'NDRG1', 'PDCD1', 'RORC', 'SCNN1A', 'SCNN1G', 'SGK1', 'SLC5A1', 'SLC9A3', 'TBP', 'TCF7', 'TOX']
x_metab   : MISSING -- re-run so the job stores it; beta*x / R^2 cells will skip.
cell types: ['B', 'Endothelial', 'Enteric_glia', 'Fibroblast', 'IEC', 'MAST', 'MNP', 'Muscularis_mucosa', 'Myofibroblast', 'Pericyte', 'Plasma', 'T']


## Diagnostic — is the run analysable, and is `T` populated?

The empty-`subsample_beta_means.csv` symptom is that the trained cells have no label under
`TIER`, so grouping drops them. Confirm coverage here before trusting the numbers below.

In [4]:
# how many trained cells carry a label under TIER (per gene, first subsample it appears in)
ct = adata.obs[TIER].astype(str)
for g in GENES[:1] or []:
    e = next(x for x in SUBIDX if x['gene'] == g)
    b = adata.obsm[e['key']]
    trained = ~np.all(np.isnan(b), axis=1) if b.shape[1] else np.zeros(adata.n_obs, bool)
    lab = ct[trained]
    print(f'{g}: trained cells={int(trained.sum())} | labeled under {TIER!r}='
          f'{int((lab != "nan").sum())}')
    print('  label counts among trained:', dict(lab.value_counts().head(10)))

n_ct = int((ct == CELL_TYPE).sum())
print(f'\ncells with cell type == {CELL_TYPE!r}: {n_ct}')
if n_ct == 0:
    print('  -> no cells of this type; pick one from the list above, or fix the annotation.')
if not HAS_X:
    print('  -> x_metab missing: sections 3 (beta*x / R^2) will be skipped.')

ATP1A1: trained cells=0 | labeled under '25_06_11_ICI_5K_Coarse_annotations'=0
  label counts among trained: {}

cells with cell type == 'T': 632
  -> x_metab missing: sections 3 (beta*x / R^2) will be skipped.


## Helpers (pure — obsm slicing only)

In [5]:
def _ct_mask(cell_type=CELL_TYPE):
    return (adata.obs[TIER].astype(str) == cell_type).to_numpy()

def _entries(gene):
    """subsample_index entries for a target gene: [(sample, key, [columns]), ...]."""
    return [(e['sample'], e['key'], list(e['columns'])) for e in SUBIDX if e['gene'] == gene]

def beta_df(gene, sample):
    """betas for (gene, sample) as a cells x gene-pair DataFrame (empty if none)."""
    for s, key, cols in _entries(gene):
        if s == sample:
            return pd.DataFrame(adata.obsm[key], index=adata.obs_names, columns=cols)
    return pd.DataFrame(index=adata.obs_names)

def raw_counts(gene):
    """Raw counts of `gene` for every cell (adata.X unless RAW_LAYER is set)."""
    xg = adata[:, gene].layers[RAW_LAYER] if RAW_LAYER else adata[:, gene].X
    return xg.toarray().ravel() if hasattr(xg, 'toarray') else np.asarray(xg).ravel()

## 1. Average β per gene pair within a cell type

One row per `(subsample, gene, gene_pair)` with the mean/std β over the chosen cell type's
cells, plus a cross-subsample summary (a pair recurs across draws whenever the Bernoulli
sample kept it). This is the correct, per-cell version of `subsample_beta_means.csv`.

In [7]:
def avg_beta_by_pair(gene, cell_type=CELL_TYPE):
    mask = _ct_mask(cell_type)
    rows = []
    for s, key, cols in _entries(gene):
        b = pd.DataFrame(adata.obsm[key], index=adata.obs_names, columns=cols).loc[mask]
        for pair in cols:
            v = b[pair].to_numpy(); v = v[~np.isnan(v)]
            if v.size:
                rows.append({'sample': s, 'gene': gene, 'gene_pair': pair,
                             'mean_beta': float(v.mean()), 'std_beta': float(v.std()),
                             'n_cells': int(v.size)})
    return pd.DataFrame(rows)

def avg_beta_summary(gene, cell_type=CELL_TYPE):
    df = avg_beta_by_pair(gene, cell_type)
    if df.empty:
        return df
    g = df.groupby('gene_pair')
    return (pd.DataFrame({'n_subsamples': g.size(),
                          'mean_beta': g['mean_beta'].mean(),
                          'std_over_subsamples': g['mean_beta'].std(),
                          'mean_abs_beta': g['mean_beta'].apply(lambda s: np.mean(np.abs(s)))})
            .reset_index().sort_values('mean_abs_beta', ascending=False, ignore_index=True))

print(f'per-(subsample, pair) means for {GENE} in {CELL_TYPE!r}:')
display(avg_beta_by_pair(GENE, CELL_TYPE).head(20))
print(f'\ncross-subsample summary for {GENE} in {CELL_TYPE!r}:')
display(avg_beta_summary(GENE, CELL_TYPE).head(20))

per-(subsample, pair) means for NDRG1 in 'T':


""



cross-subsample summary for NDRG1 in 'T':


""


In [8]:
# The run also writes this as subsample_beta_means.csv (tier_means per subsample) -- the
# "already calculated" table. It matches avg_beta_by_pair once the trained cells are labeled
# under the cell-type column. Shown here filtered to the gene + cell type.
_csv = sub_root / 'subsample_beta_means.csv'
if _csv.is_file():
    _m = pd.read_csv(_csv)
    print(f'{_csv.name}: {len(_m)} rows')
    if len(_m):
        display(_m[(_m['gene'] == GENE) &
                   (_m['cell_type'].astype(str) == CELL_TYPE)].head(20))
else:
    print('no subsample_beta_means.csv at', _csv)

subsample_beta_means.csv: 0 rows


## 2. Distribution of β for each gene pair

Per-cell β for each gene pair of the target gene, pooled over the cell type's cells and over
every subsample that kept the pair. `top` limits to the strongest pairs by mean |β|.

In [9]:
def beta_values_by_pair(gene, cell_type=CELL_TYPE):
    mask = _ct_mask(cell_type)
    pooled = {}
    for s, key, cols in _entries(gene):
        b = pd.DataFrame(adata.obsm[key], index=adata.obs_names, columns=cols).loc[mask]
        for pair in cols:
            v = b[pair].to_numpy(); v = v[~np.isnan(v)]
            if v.size:
                pooled.setdefault(pair, []).append(v)
    return {k: np.concatenate(v) for k, v in pooled.items()}

def plot_beta_distributions(gene, cell_type=CELL_TYPE, top=None, bins=60):
    pooled = beta_values_by_pair(gene, cell_type)
    if not pooled:
        print(f'no gene-pair betas for {gene} in {cell_type!r}'); return
    order = sorted(pooled, key=lambda k: -np.mean(np.abs(pooled[k])))
    if top:
        order = order[:top]
    plt.figure(figsize=(9, 5))
    for pair in order:
        plt.hist(pooled[pair], bins=bins, alpha=0.45, label=pair.replace('metab@', ''))
    plt.axvline(0, color='k', lw=0.6)
    plt.xlabel(f'{gene} beta'); plt.ylabel('cells (pooled over subsamples)')
    plt.title(f'{DATASET} | {gene} beta distribution per gene pair | {cell_type}')
    plt.legend(fontsize=7, ncol=2); plt.tight_layout(); plt.show()

plot_beta_distributions(GENE, CELL_TYPE, top=12)

no gene-pair betas for NDRG1 in 'T'


## 3. R² — cell-type raw expression vs β·x

β·x is a gene pair's contribution for the target gene (`beta[:, pair] * x_metab[:, pair]`). We
regress the cell type's **raw counts** of the gene on β·x and report R², per subsample and then
averaged across subsamples per gene pair. Needs the stored `x_metab`.

In [10]:
def _r2(a, b):
    m = ~np.isnan(a) & ~np.isnan(b)
    if m.sum() < 3 or np.ptp(a[m]) == 0 or np.ptp(b[m]) == 0:
        return np.nan, int(m.sum())
    return float(linregress(a[m], b[m]).rvalue ** 2), int(m.sum())

def betax_r2_by_pair(gene, cell_type=CELL_TYPE):
    if not HAS_X:
        print('x_metab missing -- re-run so the job stores it.'); return pd.DataFrame()
    if gene not in adata.var_names:
        print(f'{gene} not in adata.var_names'); return pd.DataFrame()
    mask = _ct_mask(cell_type)
    raw = raw_counts(gene)[mask]
    rows = []
    for s, key, cols in _entries(gene):
        b = pd.DataFrame(adata.obsm[key], index=adata.obs_names, columns=cols).loc[mask]
        for pair in cols:
            if pair not in X.columns:
                continue
            betax = b[pair].to_numpy() * X.loc[mask, pair].to_numpy()
            r2, n = _r2(betax, raw)
            rows.append({'sample': s, 'gene': gene, 'gene_pair': pair, 'r2': r2,
                         'avg_betax': float(np.nanmean(betax)),
                         'avg_abs_beta': float(np.nanmean(np.abs(b[pair].to_numpy()))),
                         'avg_x': float(np.nanmean(X.loc[mask, pair].to_numpy())),
                         'avg_raw_expr': float(np.nanmean(raw)), 'n_cells': n})
    return pd.DataFrame(rows)

def betax_r2_summary(gene, cell_type=CELL_TYPE):
    df = betax_r2_by_pair(gene, cell_type)
    if df.empty:
        return df
    g = df.groupby('gene_pair')
    return (pd.DataFrame({'n_subsamples': g.size(), 'mean_r2': g['r2'].mean(),
                          'std_r2': g['r2'].std(), 'mean_avg_betax': g['avg_betax'].mean()})
            .reset_index().sort_values('mean_r2', ascending=False, ignore_index=True))

print(f'per-(subsample, pair) R^2 for {GENE} raw counts vs beta*x in {CELL_TYPE!r}:')
display(betax_r2_by_pair(GENE, CELL_TYPE).sort_values('r2', ascending=False, ignore_index=True).head(20))
print(f'\ncross-subsample R^2 summary for {GENE} in {CELL_TYPE!r}:')
display(betax_r2_summary(GENE, CELL_TYPE).head(20))

per-(subsample, pair) R^2 for NDRG1 raw counts vs beta*x in 'T':
x_metab missing -- re-run so the job stores it.


KeyError: 'r2'

In [ ]:
# Scatter for one (gene, gene_pair): raw counts vs beta*x, per subsample, with the R^2 fit line.
def plot_betax_vs_expr(gene, gene_pair, cell_type=CELL_TYPE):
    if not HAS_X:
        print('x_metab missing.'); return
    if not gene_pair.startswith('metab@'):
        gene_pair = 'metab@' + gene_pair
    ent = [(s, key, cols) for s, key, cols in _entries(gene) if gene_pair in cols]
    if not ent:
        print(f'{gene_pair} not fit for {gene} in any subsample'); return
    mask = _ct_mask(cell_type)
    raw = raw_counts(gene)[mask]
    n = len(ent); ncol = min(3, n); nrow = -(-n // ncol)
    fig, axes = plt.subplots(nrow, ncol, figsize=(5 * ncol, 4 * nrow), squeeze=False)
    for ax, (s, key, cols) in zip(axes.ravel(), ent):
        b = pd.DataFrame(adata.obsm[key], index=adata.obs_names, columns=cols).loc[mask, gene_pair].to_numpy()
        betax = b * X.loc[mask, gene_pair].to_numpy()
        r2, nc = _r2(betax, raw)
        ax.scatter(betax, raw, s=10, alpha=0.5)
        mm = ~np.isnan(betax) & ~np.isnan(raw)
        if mm.sum() >= 3 and np.ptp(betax[mm]) > 0:
            sl, ic, *_ = linregress(betax[mm], raw[mm])
            xs = np.linspace(betax[mm].min(), betax[mm].max(), 50)
            ax.plot(xs, sl * xs + ic, 'r--', label=f'R^2={r2:.3f} (n={nc})')
            ax.legend(fontsize=8)
        ax.set_xlabel(f'{gene} beta*x'); ax.set_ylabel(f'{gene} raw counts')
        ax.set_title(f'subsample {s}')
    for ax in axes.ravel()[n:]:
        ax.axis('off')
    fig.suptitle(f'{DATASET} | {gene} ~ {gene_pair.replace("metab@", "")} | {cell_type}')
    fig.tight_layout(rect=[0, 0, 1, 0.96]); plt.show()

_summ = betax_r2_summary(GENE, CELL_TYPE)
if not _summ.empty:
    plot_betax_vs_expr(GENE, _summ.iloc[0]['gene_pair'], CELL_TYPE)